# <b> pasar de “cuánto gano en cada escenario” a “qué resultado espero considerando sus probabilidades”.</b>

- Usa el straddle y el strangle del ejercicio anterior, conservando contratos y costes. La pregunta será: ¿la estructura con mayor probabilidad de ganar también tiene mayor ganancia esperada?

Trabaja con estos dos conjuntos de probabilidades inventadas para aprender; no son pronósticos de SPY:

| Cambio de SPY hasta vencimiento | Distribución A | Distribución B |
|---|---:|---:|
| −10% | 10% | 20% |
| −5% | 20% | 15% |
| 0% | 40% | 30% |
| +5% | 20% | 15% |
| +10% | 10% | 20% |

B asigna más probabilidad a movimientos extremos, manteniendo el mismo retorno esperado de cero.

Para cada estructura y distribución calcula:
1. Probabilidad de ganar: suma de probabilidades de escenarios con resultado estrictamente positivo.
2. Ganancia esperada: $ \mathbb{E}[\Pi]=\sum_s p_s\Pi_s $ 
3. Pérdida máxima contractual, y qué porcentaje de la prima representa la ganancia esperada.
Después responde:
  - Alguna estructura gana frecuentemente pero tiene ganancia esperada negativa?
  - ¿Cómo cambia su atractivo al pasar de A a B?
  - ¿Puedes elegir entre ellas sin tener una creencia sobre los movimientos futuros?

El aprendizaje económico: conocer el payoff y la prima permite describir una operación; evaluar su atractivo exige incorporar una distribución de resultados. Esto conecta directamente con los escenarios y probabilidades de tu motor de portafolio.

In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import date
import gzip
import json
import pandas as pd

# Lista editable para el ejercicio.
exercise_dates = ['2020-01-02', '2022-10-03', '2025-04-03', '2026-08-31']

cwd = Path.cwd().resolve()
repo_root = next((parent for parent in (cwd, *cwd.parents) if (parent / 'quant-research-notes').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('No se encontró la raíz del repositorio: se esperaba una carpeta quant-research-notes.')

hmm_root = repo_root / 'quant-research-notes' / 'HMM_regime_options'
raw_dir = hmm_root / 'data' / 'raw'
price_path = hmm_root / 'data' / 'derived' / 'spy_daily_adjusted_2020_2026.csv'
if not raw_dir.is_dir():
    raise FileNotFoundError(f'No existe el directorio de cadenas originales: {raw_dir}')
if not price_path.is_file():
    raise FileNotFoundError(f'No existe la tabla local de cierres SPY: {price_path}')

prices = pd.read_csv(price_path, parse_dates=['date']).set_index('date')
if 'close' not in prices.columns:
    raise KeyError(f'Falta el campo close en {price_path}')

def load_chain(chain_date):
    candidates = [raw_dir / f'spy_historical_options_{chain_date}.json.gz', raw_dir / f'spy_historical_options_{chain_date}.json']
    chain_path = next((path for path in candidates if path.is_file()), None)
    if chain_path is None:
        raise FileNotFoundError(f'Falta la cadena original de SPY para {chain_date}; se buscó .json.gz y .json en {raw_dir}')
    opener = gzip.open if chain_path.suffix == '.gz' else open
    with opener(chain_path, 'rt', encoding='utf-8') as handle:
        payload = json.load(handle)
    records = payload.get('data')
    if not isinstance(records, list):
        raise KeyError(f'La cadena {chain_path.name} no contiene una lista data de contratos')
    return chain_path, records

work_frames = []
required_fields = {'contractID', 'type', 'expiration', 'strike', 'bid', 'ask'}
for chain_date in exercise_dates:
    observation_date = pd.Timestamp(chain_date)
    if observation_date not in prices.index:
        raise KeyError(f'No hay cierre SPY para {chain_date} en {price_path.name}')
    spy_close = prices.loc[observation_date, 'close']
    chain_path, raw_records = load_chain(chain_date)

    grouped = defaultdict(list)
    for record in raw_records:
        missing = required_fields.difference(record)
        if missing:
            raise KeyError(f'Faltan campos {sorted(missing)} en la cadena {chain_path.name}; contractID={record.get("contractID")}')
        grouped[str(record['contractID'])].append(record)
    deduplicated = []
    for contract_id, records in grouped.items():
        fingerprints = {json.dumps(record, sort_keys=True, separators=(',', ':'), ensure_ascii=False) for record in records}
        if len(fingerprints) > 1:
            raise RuntimeError(f'Conflicto: contractID {contract_id} tiene registros distintos en {chain_path.name}; el ejercicio se detuvo.')
        deduplicated.append(records[0])

    expiration_sides = defaultdict(set)
    for record in deduplicated:
        try:
            expiration = date.fromisoformat(str(record['expiration']))
            dte = (expiration - observation_date.date()).days
        except ValueError as exc:
            raise ValueError(f'Vencimiento inválido en {chain_path.name}: {record["expiration"]}') from exc
        if 27 <= dte <= 33 and record['type'] in {'call', 'put'}:
            expiration_sides[str(record['expiration'])].add(record['type'])
    common_expirations = [expiration for expiration, sides in expiration_sides.items() if sides == {'call', 'put'}]
    if not common_expirations:
        raise ValueError(f'No hay vencimiento común call/put entre 27 y 33 DTE para {chain_date} en {chain_path.name}')
    selected_expiration = min(
        common_expirations,
        key=lambda expiration: (abs((date.fromisoformat(expiration) - observation_date.date()).days - 30), expiration),
    )
    selected_dte = (date.fromisoformat(selected_expiration) - observation_date.date()).days
    rows = []
    for record in deduplicated:
        if record['type'] not in {'call', 'put'} or str(record['expiration']) != selected_expiration:
            continue
        try:
            strike = float(record['strike'])
        except (TypeError, ValueError) as exc:
            raise ValueError(f'Strike inválido para {record["contractID"]} en {chain_path.name}') from exc
        if abs(strike / float(spy_close) - 1.0) > 0.05:
            continue
        bid_number = pd.to_numeric(record['bid'], errors='coerce')
        ask_number = pd.to_numeric(record['ask'], errors='coerce')
        rows.append({
            'date': chain_date, 'type': record['type'], 'expiration': selected_expiration, 'DTE': selected_dte,
            'strike': strike, 'SPY_close': float(spy_close), 'bid': record['bid'], 'ask': record['ask'],
            'mid': (bid_number + ask_number) / 2, 'contractID': record['contractID'],
        })
    work_frames.append(pd.DataFrame(rows))

options_work = pd.concat(work_frames, ignore_index=True)
options_work = options_work[['date', 'type', 'expiration', 'DTE', 'strike', 'SPY_close', 'bid', 'ask', 'mid', 'contractID']]
options_work = options_work.sort_values(['date', 'type', 'strike', 'contractID']).reset_index(drop=True)
display(options_work)
print('Filas por fecha:')
display(options_work.groupby('date').size().rename('rows').reset_index())

,date,type,expiration,DTE,strike,SPY_close,bid,ask,mid,contractID
0,2020-01-02,call,2020-01-31,29,309.0,324.87,17.19,17.30,17.245,SPY200131C00309000
1,2020-01-02,call,2020-01-31,29,309.5,324.87,16.73,16.87,16.800,SPY200131C00309500
2,2020-01-02,call,2020-01-31,29,310.0,324.87,16.27,16.39,16.330,SPY200131C00310000
3,2020-01-02,call,2020-01-31,29,310.5,324.87,15.81,15.94,15.875,SPY200131C00310500
4,2020-01-02,call,2020-01-31,29,311.0,324.87,15.35,15.48,15.415,SPY200131C00311000
...,...,...,...,...,...,...,...,...,...,...
415,2026-08-31,put,2026-09-30,30,801.0,767.05,32.59,36.10,34.345,SPY260930P00801000
416,2026-08-31,put,2026-09-30,30,802.0,767.05,33.53,37.05,35.290,SPY260930P00802000
417,2026-08-31,put,2026-09-30,30,803.0,767.05,34.49,38.00,36.245,SPY260930P00803000
418,2026-08-31,put,2026-09-30,30,804.0,767.05,35.44,38.95,37.195,SPY260930P00804000


Filas por fecha:


,date,rows
0,2020-01-02,112
1,2022-10-03,70
2,2025-04-03,84
3,2026-08-31,154


exactamente los contratos usados en el ejercicio 05 y vuelve a calcular sus costes desde sus ask.

In [2]:
# Estructuras conservadas del ejercicio 05.

straddle_ids = [
    "SPY260930C00767000",
    "SPY260930P00767000",
]

strangle_ids = [
    "SPY260930C00790000",
    "SPY260930P00744000",
]

straddle = options_work[
    options_work["contractID"].isin(straddle_ids)
].sort_values("type")

strangle = options_work[
    options_work["contractID"].isin(strangle_ids)
].sort_values("type")

costo_straddle = straddle["ask"].astype(float).sum()
costo_strangle = strangle["ask"].astype(float).sum()

display(straddle)
display(strangle)

print(f"Coste del straddle: ${costo_straddle:.2f} por acción")
print(f"Coste del strangle: ${costo_strangle:.2f} por acción")

,date,type,expiration,DTE,strike,SPY_close,bid,ask,mid,contractID
304,2026-08-31,call,2026-09-30,30,767.0,767.05,10.82,10.87,10.845,SPY260930C00767000
381,2026-08-31,put,2026-09-30,30,767.0,767.05,10.03,10.08,10.055,SPY260930P00767000


,date,type,expiration,DTE,strike,SPY_close,bid,ask,mid,contractID
327,2026-08-31,call,2026-09-30,30,790.0,767.05,1.91,1.93,1.92,SPY260930C00790000
358,2026-08-31,put,2026-09-30,30,744.0,767.05,4.16,4.20,4.18,SPY260930P00744000


Coste del straddle: $20.95 por acción
Coste del strangle: $6.13 por acción


### Construimos los cinco escenarios y calculamos el resultado de cada estructura. Todavía no calculamos probabilidades de ganar ni ganancia esperada.

In [3]:
# Escenarios hipotéticos y resultados al vencimiento.

spot = straddle["SPY_close"].iloc[0]

K_straddle = straddle["strike"].iloc[0]
K_put_strangle = strangle[strangle["type"] == "put"]["strike"].iloc[0]
K_call_strangle = strangle[strangle["type"] == "call"]["strike"].iloc[0]

escenarios = pd.DataFrame({
    "cambio_SPY": [-0.10, -0.05, 0, 0.05, 0.10],
    "probabilidad_A": [0.10, 0.20, 0.40, 0.20, 0.10],
    "probabilidad_B": [0.20, 0.15, 0.30, 0.15, 0.20],
})

escenarios["SPY_al_vencimiento"] = spot * (1 + escenarios["cambio_SPY"])

# Resultado del straddle: call ATM + put ATM menos su coste.
escenarios["resultado_straddle"] = (
    (escenarios["SPY_al_vencimiento"] - K_straddle).clip(lower=0)
    + (K_straddle - escenarios["SPY_al_vencimiento"]).clip(lower=0)
    - costo_straddle
)

# Resultado del strangle: call OTM + put OTM menos su coste.
escenarios["resultado_strangle"] = (
    (escenarios["SPY_al_vencimiento"] - K_call_strangle).clip(lower=0)
    + (K_put_strangle - escenarios["SPY_al_vencimiento"]).clip(lower=0)
    - costo_strangle
)

display(escenarios)

,cambio_SPY,probabilidad_A,probabilidad_B,SPY_al_vencimiento,resultado_straddle,resultado_strangle
0,-0.10,0.1,0.20,690.3450,55.7050,47.5250
1,-0.05,0.2,0.15,728.6975,17.3525,9.1725
2,0.00,0.4,0.30,767.0500,-20.9000,-6.1300
3,0.05,0.2,0.15,805.4025,17.4525,9.2725
4,0.10,0.1,0.20,843.7550,55.8050,47.6250


## Siguiente paso — Ponderar los resultados por sus probabilidades

Ya conocemos cuánto gana o pierde cada estructura en cada escenario. Ahora usamos las probabilidades de A y B para responder dos preguntas distintas:

$ \Pr(\Pi>0)=\sum p_s \text{ para los escenarios con resultado positivo} $

$ \mathbb{E}[\Pi]=\sum_s p_s\Pi_s $

La probabilidad de ganar cuenta cuántas veces esperamos terminar con resultado positivo. La ganancia esperada también considera cuánto se gana o se pierde en cada escenario.

Calcularemos ambas medidas para el straddle y el strangle, bajo las distribuciones A y B.

In [ ]:
# Probabilidad de ganar y ganancia esperada bajo A y B.

resumen = []

estructuras = {
    "straddle": ("resultado_straddle", costo_straddle),
    "strangle": ("resultado_strangle", costo_strangle),
}

distribuciones = {
    "A": "probabilidad_A",
    "B": "probabilidad_B",
}

for estructura, (columna_resultado, costo) in estructuras.items():
    for distribucion, columna_probabilidad in distribuciones.items():

        resultado = escenarios[columna_resultado]
        probabilidad = escenarios[columna_probabilidad]

        probabilidad_ganar = probabilidad[resultado > 0].sum()
        ganancia_esperada = (probabilidad * resultado).sum()

        resumen.append({
            "estructura": estructura,
            "distribución": distribucion,
            "probabilidad_de_ganar": probabilidad_ganar,
            "ganancia_esperada": ganancia_esperada,
            "pérdida_máxima": -costo,
            "ganancia_esperada_pct_prima": ganancia_esperada / costo,
        })

resumen = pd.DataFrame(resumen)

display(resumen.style.format({
    "probabilidad_de_ganar": "{:.0%}",
    "ganancia_esperada": "${:,.2f}",
    "pérdida_máxima": "${:,.2f}",
    "ganancia_esperada_pct_prima": "{:.1%}",
}))

,estructura,distribución,probabilidad_de_ganar,ganancia_esperada,pérdida_máxima,ganancia_esperada_pct_prima
0,straddle,A,60%,$9.75,$-20.95,46.5%
1,straddle,B,70%,$21.25,$-20.95,101.4%
2,strangle,A,60%,$10.75,$-6.13,175.4%
3,strangle,B,70%,$19.96,$-6.13,325.6%


<b> Compara, lado a lado, la probabilidad de ganar y la ganancia esperada para cada estructura bajo las distribuciones A y B. </b>

In [5]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Probabilidad de ganar",
        "Ganancia esperada por acción",
    ),
)

colores = {"A": "#1f77b4", "B": "#ff7f0e"}

for distribucion in ["A", "B"]:
    datos = resumen[resumen["distribución"] == distribucion]

    fig.add_trace(
        go.Bar(
            x=datos["estructura"],
            y=datos["probabilidad_de_ganar"],
            name=f"Distribución {distribucion}",
            marker_color=colores[distribucion],
            text=datos["probabilidad_de_ganar"].map(lambda x: f"{x:.0%}"),
            textposition="auto",
            legendgroup=distribucion,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=datos["estructura"],
            y=datos["ganancia_esperada"],
            name=f"Distribución {distribucion}",
            marker_color=colores[distribucion],
            text=datos["ganancia_esperada"].map(lambda x: f"${x:.2f}"),
            textposition="auto",
            legendgroup=distribucion,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.add_hline(y=0, line_color="black", row=1, col=2)

fig.update_yaxes(tickformat=".0%", title="Probabilidad", row=1, col=1)
fig.update_yaxes(title="USD por acción", row=1, col=2)

fig.update_layout(
    title="Probabilidad de ganar frente a ganancia esperada",
    barmode="group",
    template="plotly_white",
)

fig.show()

Las barras naranjas representan la distribución B. Son más altas porque B asigna más probabilidad a movimientos extremos de SPY, de \(-10\%\) y \(+10\%\).

Eso favorece a ambas estructuras compradas:

- En la gráfica de la izquierda, una barra naranja más alta significa que la estructura tiene mayor probabilidad de terminar con resultado positivo bajo B.
- En la gráfica de la derecha, una barra naranja más alta significa mayor ganancia esperada bajo B.

La razón económica es que tanto el straddle como el strangle comprados ganan cuando SPY se mueve mucho, en cualquier dirección. B reduce la probabilidad del escenario central de \(0\%\), donde ambas estructuras pierden la prima, y aumenta la de escenarios extremos, donde ganan.

No significa que B sea “mejor” en la realidad: B es solo una creencia inventada sobre los movimientos futuros. Si crees que SPY se moverá poco, A —o una distribución aún más concentrada cerca de cero— sería más relevante y las opciones compradas resultarían menos atractivas.